# Pipeline to streamline the following tasks

 1. Extract YT video urls to create a curated dataset  
 >[VIDEO_ID, VIDEO_TITLE, VIDEO_URL, Duration, Channel]  

 2. Extract audios from the videos using URLS or video_IDs.
 >Strip the .mp3 codec audio files from the videos and save them as a dataset.
 >> DataSet > VideoID > [audio.mp3]

 3. Establish process to extract / generate transcriptions from the videos.
 >Likely use of AI models for transcription due to no presence of youtube generated transcripts in the videos.

##  Module Installation

In [ ]:
%%capture
!pip install yt-dlp -U youtube-transcript-api pydub
!apt-get update
!apt-get install -y ffmpeg

## Module Imports

In [ ]:
import yt_dlp
import pandas as pd
import youtube_transcript_api

### Example Usage: Get Channel Link from Title

Enter the channel title below to find its YouTube link. You can modify the `channel_title_to_find` variable.

In [ ]:
# channel_title_to_find = "T-Series"  # Replace with the actual channel title you want to search for

# found_channel_link = get_channel_link_from_title(channel_title_to_find)

# if found_channel_link:
#     print(f"\nThe YouTube link for '{channel_title_to_find}' is: {found_channel_link}")
# else:
#     print(f"\nNo YouTube link found for '{channel_title_to_find}'. Please check the title.")

## Main Variables

In [ ]:
video_channel = []
video_IDs = []
video_durations = []
video_language = []
videos_LARGE = []
video_titles = []
video_language_INFO = {
    'Hindi'                                     : "hi-IN",
    'Kashmiri'                                  : "ks-IN",
    'Assamese'                                  : "as-IN",
    "Marathi"                                   : "mr-IN",
    "Gujarati"                                  : "gu-IN",
    "Kannada"                                   : "kn-IN",
    "Odia"                                      :"od-IN",
    "Bengali"                                   :"bn-IN"
}

### Constant for yt_dlp

In [ ]:
ydl_opts = {
                'extract_flat': True,
                'skip_download': True,
                'ignoreerrors': True,
                # 'http_headers': {
                #     'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
                #     'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
                #     'Accept-Language': 'en-us,en;q=0.5',
                #     'Sec-Fetch-Mode': 'navigate'
                # },
                # 'geo_bypass_country': 'US' # Or any other country code
            }

In [ ]:
#Helper Functions

def Larger_Video_CHECK(VIDEO_ID: str, DURATION: int):
    """Helper function to move larger videos to another area for processing later on"""
    if(DURATION > 1800):
        videos_LARGE.append(VIDEO_ID)
        return True

    return False


def YT_Link_Generator(VIDEO_ID :str) -> str:
    """Generator Function to generate YT_Link from VIDEO_ID"""
    return "https://www.youtube.com/watch?v=" + VIDEO_ID



## Get YT Channel Usernames and Playlists

In [ ]:
Sources = []

print("Please Provide the complete clean links")
def Get_Sources():
    while(1):
        print("Add the sources from which you want to source the YT Videos(Press 'N'/'n' to stop.):")
        a = input().strip()
        if(a in ['N','n']):
            break;

        if a=='': continue

        Sources.append(a)

Get_Sources()

Please Provide the complete clean links
Add the sources from which you want to source the YT Videos(Press 'N'/'n' to stop.):
https://www.youtube.com/playlist?list=PLUiMfS6qzIMxC3SpknSSQ85dhfyPEz-rb
Add the sources from which you want to source the YT Videos(Press 'N'/'n' to stop.):
n


### Function to fetch and extract video details to formulate the csv dataset

In [ ]:
### Start Video ID, duration, extraction

def extract_video_details(Video_URL : str):

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(Video_URL, download=False)

        for entry in info['entries']:
            video_channel.append(info['channel'])
            video_IDs.append(entry['id'])
            video_durations.append(entry['duration'])
            video_titles.append(entry['title'])

In [ ]:
%%capture
for source in Sources:
    extract_video_details(source)



In [ ]:
print(len(video_channel) == len(video_durations) ==len(video_IDs))

True


### Form the .csv Dataset

In [ ]:
##Form csv dataset
#.csv dataset creation with [SrNo. , YT_VIDEO_ID, Duration, CHANNEL]
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "YT_VIDEO_ID": video_IDs,
    "YT_VIDEO_TITLE": video_titles,
    "YT_VIDEO_LINK": [YT_Link_Generator(ID) for ID in video_IDs],
    "Duration": video_durations,
    "Channel": video_channel
})

df.to_csv("YT_DATASET.csv", index = False)

### DOWNLOAD AUDIO

In [ ]:
# %%capture #Comment out if you dont want any outputs
def Don_Eladio(video_ID, video_Duration):
    flag = Larger_Video_CHECK(video_ID, video_Duration)

    if(flag): return

    ydl_opts = {
        'format': 'bestaudio/best',
        'postprocessors': [{
            'key': 'FFmpegExtractAudio',
            'preferredcodec': 'mp3',
            'preferredquality': '192',
        }],
        'outtmpl': f'KrishiDarshan/{video_ID}/audio_{video_Duration}.%(ext)s',
        # 'http_headers': {
        #     'User-Agent':| 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
        #     'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
        #     'Accept-Language': 'en-us,en;q=0.5',
        #     'Sec-Fetch-Mode': 'navigate'
        # },
        # 'geo_bypass_country': 'US' # Or any other country code
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([YT_Link_Generator(video_ID)])


# l = len(video_IDs)
# for i in range(l):
#     Don_Eladio(video_IDs[i], video_durations[i])


l = len(video_IDs[:2])
for i in range(l):
    Don_Eladio(video_IDs[i], video_durations[i])

[youtube] Extracting URL: https://www.youtube.com/watch?v=SxgaGXcQ8ZY
[youtube] SxgaGXcQ8ZY: Downloading webpage


[youtube] SxgaGXcQ8ZY: Downloading android vr player API JSON
[info] SxgaGXcQ8ZY: Downloading 1 format(s): 140
[download] Destination: KrishiDarshan/SxgaGXcQ8ZY/audio_1377.m4a
[download] 100% of   20.85MiB in 00:00:03 at 6.72MiB/s   
[FixupM4a] Correcting container of "KrishiDarshan/SxgaGXcQ8ZY/audio_1377.m4a"
[ExtractAudio] Destination: KrishiDarshan/SxgaGXcQ8ZY/audio_1377.mp3
Deleting original file KrishiDarshan/SxgaGXcQ8ZY/audio_1377.m4a (pass -k to keep)
[youtube] Extracting URL: https://www.youtube.com/watch?v=JLwhLr4ZVd0
[youtube] JLwhLr4ZVd0: Downloading webpage


[youtube] JLwhLr4ZVd0: Downloading android vr player API JSON
[info] JLwhLr4ZVd0: Downloading 1 format(s): 140
[download] Destination: KrishiDarshan/JLwhLr4ZVd0/audio_1472.m4a
[download] 100% of   22.29MiB in 00:00:08 at 2.66MiB/s   
[FixupM4a] Correcting container of "KrishiDarshan/JLwhLr4ZVd0/audio_1472.m4a"
[ExtractAudio] Destination: KrishiDarshan/JLwhLr4ZVd0/audio_1472.mp3
Deleting original file KrishiDarshan/JLwhLr4ZVd0/audio_1472.m4a (pass -k to keep)


In [ ]:
# Zip audio files as a dataset

!zip -r KrishiDarshan.zip KrishiDarshan

  adding: KrishiDarshan/ (stored 0%)
  adding: KrishiDarshan/SxgaGXcQ8ZY/ (stored 0%)
  adding: KrishiDarshan/SxgaGXcQ8ZY/audio_1377.mp3 (deflated 1%)
  adding: KrishiDarshan/JLwhLr4ZVd0/ (stored 0%)
  adding: KrishiDarshan/JLwhLr4ZVd0/audio_1472.mp3 (deflated 1%)


In [ ]:
# %%capture
# for i in range(len(video_IDs[:1])):
#     flag = Larger_Video_CHECK(video_IDs[i], video_durations[i])

#     if(flag): continue

#     ydl_opts = {
#         'format': 'bestaudio/best',
#         'postprocessors': [{
#             'key': 'FFmpegExtractAudio',
#             'preferredcodec': 'mp3',
#             'preferredquality': '192',
#         }],
#         'outtmpl': f'KrishiDarshan/{video_IDs[i]}/audio.%(ext)s',
#     }

#     with yt_dlp.YoutubeDL(ydl_opts) as ydl:
#         ydl.download([YT_Link_Generator(video_IDs[i])])

### Start Generating the transcript using whisper large v3 from Hugging Face

> Implement a Batch Processing Architecture to process the Audio files for highest possible accuracy in the transcripts.

In [ ]:
# WHisper works best with 30 second splits of the audio files fed to the api batch after batch

### The splitter will work to split all the splits first and then start transcription successively.
### Transcripts of each split will be added into the

In [ ]:
!mkdir splits

In [ ]:
from pydub import AudioSegment

def splitter(video_Duration):
    # there is an edge case here which will only occur when an audio file is smaller than 30 seconds
    """Take the audio file duration, and return a list of timestamps with seperation of 30"""
    if (video_Duration < 30):
        return

    return [i+(video_Duration - i) if (video_Duration - i) <= 29 else i  for i in range(0,video_Duration+1, 30)]

# print(splitter(59))

def audio_splitter(audio_file, start, end): # start and end must be in milliseconds
    audio = AudioSegment.from_file(audio_file)
    print(len(audio))

    segment = audio[start:end]
    segment.export(f"/content/splits/segment{start}_{end}.mp3", format='mp3')

/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


In [ ]:
audio_splitter(audio_file_paths[0], 0, 30000)

NameError: name 'audio_file_paths' is not defined

In [ ]:
# # Fetch the audio files and start transcription
# # Extract Zip file to datasets

# #Currently Running on Colab

# !unzip /content/KrishiDarshan.zip -d /content/

In [ ]:
import os

root_dir = '/content/KrishiDarshan'
audio_file_paths = []
audio_durations = []

for dirpath, dirnames, filenames in os.walk(root_dir):
    for filename in filenames:
        if filename.endswith('.mp3'):
            audio_file_paths.append(os.path.join(dirpath, filename))
            audio_durations.append(int(audio_file_paths[-1].split('_')[1].split('.')[0]))  ## must convert to integer

print(f"Found {len(audio_file_paths)} audio files:")
for path in audio_file_paths:
    print(path)


Found 2 audio files:
/content/KrishiDarshan/SxgaGXcQ8ZY/audio_1377.mp3
/content/KrishiDarshan/JLwhLr4ZVd0/audio_1472.mp3


In [ ]:
Transcripts = []
transcript = ""

In [ ]:
%%capture
!pip install --upgrade transformers

In [ ]:
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline
from datasets import load_dataset


device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

model_id = "openai/whisper-large-v3"

model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_id, torch_dtype=torch_dtype, low_cpu_mem_usage=True, use_safetensors=True
)
model.to(device)

processor = AutoProcessor.from_pretrained(model_id)

pipe = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    torch_dtype=torch_dtype,
    device=device,
)


config.json:   0%|          | 0.00/1.27k [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.90k [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/494k [00:00<?, ?B/s]

normalizer.json:   0%|          | 0.00/52.7k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/34.6k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.07k [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


In [ ]:
print(splitter(1472))

In [ ]:
#current sample only contains 2 audios

for file in audio_file_paths:
    splits = splitter(audio_durations[0])
    transcript = ""

    for i in range(1, len(splits)+1):
        segment = audio_splitter(file, splits[i-1]*1000, splits[i]*1000)

    break

1376340
1376340
1376340
1376340
1376340
1376340
1376340
1376340
1376340
1376340
1376340
1376340
1376340
1376340
1376340
1376340
1376340
1376340
1376340
1376340
1376340
1376340
1376340
1376340
1376340
1376340
1376340
1376340
1376340
1376340
1376340
1376340
1376340
1376340
1376340
1376340
1376340
1376340
1376340
1376340
1376340
1376340
1376340
1376340
1376340


IndexError: list index out of range

In [ ]:
audio_durations[0]

1377

In [ ]:
splits = splitter(audio_durations[0])
TRANSCRIPTS = []
transcript = ""

for dirpath, dirnames, filenames in os.walk(root_dir):
    for filename in filenames:
        if filename.endswith('.mp3'):
            for i in range(len(splits)):
                s = pipe(os.path.join(dirpath, filename), language='hi', return_timestamps = True)



                print(s['text'])

            break



[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


 किसान बहनोर भाईयों नमस्कार पिशी दर्शन कारेकरम में आप सबका स्वागत है हम सब जानते हैं कि बढ़ती जन संख्या की खादे जरूरतों को पूरा करने के लिए हमें भरपूर सतत उत्पादन चाहिए और ये भी जानते हैं कि कृषी जोत हमारी छोटी होती जा रही है और हमारे देश में छोटे तथा सीमान किसानों की संख्या जादा है कैसे हम कृषी लागत को कम करके भरपूर उत्पादन ले पोशन का आजीविका का सतत प्रबंधन हो इसके लिए हमारे कृषी विज्ञानिकों ने एक अत्यंत कारगर तकनीक विकसित की है समन्वित कृषी प्रणाली यानि एक हेक्टेर भूमी में आप सतत आजीविका ले सकते हैं, कृषि लागत को कम कर सकते हैं, पशु पालन कर सकते हैं और वो सब एक चक्र के अनुसार एक दूसरे पर निर्भर होकर भरपूर उत्पादन की आपूर्थी करते हैं तो क्या है ये समन्वित कृषि प्रणाली, कैसे इसका प्रबंधन होता है और ये कैसे किसानों के लिए मददगार है, आए लेते हैं जानकारे एक हेक्टर जमीन में सम्मनित कृषि प्रणाली दोरा कैसे एक छोटा कृषक, छोटा किसान एक हजार उपिया प्रति दिन सुद्धाय कर सकता है और उसका काम करने का है तू कैसे एक हजार उपिया वो ले सकता है अतत एक मैना में कैसे पचास से साठ हजार उपिया कमा सकता है यहां 

In [ ]:
result = pipe('/content/segment.mp3', language='hi')
print(result["text"])